# Field Explorer

This notebook covers two things:
1. **How to generate new fields** — syntax + examples for all three templates
2. **Interactive visualization** — plotly heatmaps for perm, poro, and well mask

All fields live in `fields/` as HDF5 files. Generate them with the CLI, then load and inspect here.

## 1. Generating fields

The CLI is `src/field_generator/generate_field.py`. It takes a YAML config and writes a `.h5` file.

### Template: `homogeneous`
Uniform perm and poro throughout. Baseline for all physics cases.

```yaml
# field_configs/homogeneous_100md.yaml
template: homogeneous
output: fields/homogeneous_100md_phi025.h5

grid:
  nx: 50
  ny: 50
  nz: 1

well:
  i: 25   # 0-indexed center cell
  j: 25
  k: 0

params:
  perm_md: 100.0   # milliDarcy
  poro: 0.25
```

### Template: `fourier`
Spectrally-synthesized log-normal perm field with spatial correlation. Good for "realistic but controlled" heterogeneity.

Key params:
- `perm_mean_md` — geometric mean permeability
- `perm_std_log` — heterogeneity level: 0.5=mild, 1.0=moderate, 2.0=strong
- `correlation_length` — patch size as fraction of domain (0.2 = ~400m patches on 2000m domain)
- `seed` — for reproducibility

```yaml
# field_configs/fourier_moderate.yaml
template: fourier
output: fields/fourier_moderate.h5

grid:
  nx: 50
  ny: 50
  nz: 1

well:
  i: 25
  j: 25
  k: 0

params:
  perm_mean_md: 100.0
  perm_std_log: 1.0        # moderate: P10/P90 span ~7x
  poro_mean: 0.22
  poro_std: 0.04
  correlation_length: 0.2
  seed: 42
```

### Template: `gmm`
Facies-based field using Truncated Gaussian Simulation. Each facies has its own perm distribution; weights control facies proportion.

```yaml
# field_configs/gmm_3facies.yaml
template: gmm
output: fields/gmm_3facies.h5

grid:
  nx: 50
  ny: 50
  nz: 1

well:
  i: 25
  j: 25
  k: 0

params:
  correlation_length: 0.25
  poro_std: 0.03
  seed: 42
  facies:
    - weight: 0.25           # tight shale / mudstone
      perm_mean_md: 0.5
      perm_std_log: 0.5
      poro_mean: 0.08
    - weight: 0.50           # siltstone / moderate reservoir
      perm_mean_md: 80.0
      perm_std_log: 0.8
      poro_mean: 0.20
    - weight: 0.25           # clean sandstone / good reservoir
      perm_mean_md: 600.0
      perm_std_log: 1.0
      poro_mean: 0.30
```

### Running the CLI

```bash
# From repo root:
python src/field_generator/generate_field.py field_configs/homogeneous_100md.yaml
python src/field_generator/generate_field.py field_configs/fourier_moderate.yaml
python src/field_generator/generate_field.py field_configs/gmm_3facies.yaml
```

Or run the cell below to regenerate all three from within this notebook.

In [1]:
import os, subprocess, sys
from pathlib import Path

# Ensure all repo-relative paths work regardless of where Jupyter started
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(REPO_ROOT)
print(f'Working directory: {Path.cwd()}')

PYTHON = sys.executable
CLI = 'src/field_generator/generate_field.py'
CONFIGS = [
    'field_configs/homogeneous_100md.yaml',
    'field_configs/fourier_moderate.yaml',
    'field_configs/gmm_3facies.yaml',
]

for cfg in CONFIGS:
    result = subprocess.run([PYTHON, CLI, cfg], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)

Working directory: c:\Users\zchen\workplace\dst_well_testing
Saved: fields\homogeneous_100md_phi025.h5
  shape (nz,ny,nx): (1, 50, 50)
  perm  min=100 mD  max=100 mD  geomean=100 mD
  poro  min=0.250  mean=0.250  max=0.250
  well  1 cell(s) marked

Saved: fields\fourier_moderate.h5
  shape (nz,ny,nx): (1, 50, 50)
  perm  min=12.5 mD  max=647 mD  geomean=100 mD
  poro  min=0.140  mean=0.220  max=0.328
  well  1 cell(s) marked

Saved: fields\gmm_3facies.h5
  shape (nz,ny,nx): (1, 50, 50)
  perm  min=0.089 mD  max=1.62e+04 mD  geomean=31 mD
  poro  min=0.030  mean=0.194  max=0.407
  well  1 cell(s) marked



## 2. Loading fields

In [2]:
import h5py
import numpy as np
from pathlib import Path

MD_TO_M2 = 9.869233e-16

def load_field(path):
    """Load perm (mD), poro, well_mask from an .h5 field file. Returns 2D arrays (layer 0)."""
    with h5py.File(path, 'r') as f:
        perm_m2 = f['perm'][0]          # (ny, nx)
        poro    = f['poro'][0]           # (ny, nx)
        well    = f['well_mask'][0]      # (ny, nx)
    perm_md = perm_m2 / MD_TO_M2
    return perm_md, poro, well

FIELDS = {
    'homogeneous_100md': 'fields/homogeneous_100md_phi025.h5',
    'fourier_moderate':  'fields/fourier_moderate.h5',
    'gmm_3facies':       'fields/gmm_3facies.h5',
}

data = {name: load_field(path) for name, path in FIELDS.items()}

for name, (perm, poro, well) in data.items():
    ny, nx = perm.shape
    print(f'{name:25s}  perm {perm.min():.2g}–{perm.max():.2g} mD  '
          f'poro {poro.min():.3f}–{poro.max():.3f}  '
          f'well cells: {well.sum()}')

homogeneous_100md          perm 1e+02–1e+02 mD  poro 0.250–0.250  well cells: 1
fourier_moderate           perm 12–6.5e+02 mD  poro 0.140–0.328  well cells: 1
gmm_3facies                perm 0.089–1.6e+04 mD  poro 0.030–0.407  well cells: 1


## 3. Interactive visualization

Use the dropdown to switch fields and the radio buttons to switch between perm and poro. Hover over any cell to see its value. The well location is marked with a red X.

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

def make_figure(field_name):
    perm, poro, well = data[field_name]
    ny, nx = perm.shape
    wy, wx = np.where(well)

    # x/y axis labels in meters (2000m domain)
    x_m = np.linspace(0, 2000, nx)
    y_m = np.linspace(0, 2000, ny)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Permeability (mD) — log scale', 'Porosity'],
        horizontal_spacing=0.12,
    )

    # --- permeability (log scale) ---
    log_perm = np.log10(np.clip(perm, 1e-4, None))
    vmin, vmax = log_perm.min(), log_perm.max()
    if vmin == vmax:
        vmin, vmax = vmin - 0.1, vmax + 0.1

    hover_perm = np.array(
        [[f'x={x_m[j]:.0f}m  y={y_m[i]:.0f}m<br>k = {perm[i,j]:.2g} mD'
          for j in range(nx)] for i in range(ny)]
    )

    fig.add_trace(go.Heatmap(
        z=log_perm,
        x=x_m, y=y_m,
        colorscale='Viridis',
        zmin=vmin, zmax=vmax,
        colorbar=dict(
            title='log₁₀(k / mD)',
            tickvals=[v for v in np.linspace(vmin, vmax, 5)],
            ticktext=[f'{10**v:.2g}' for v in np.linspace(vmin, vmax, 5)],
            x=0.44,
        ),
        hovertext=hover_perm,
        hoverinfo='text',
        name='perm',
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=x_m[wx], y=y_m[wy],
        mode='markers',
        marker=dict(symbol='x', size=14, color='red', line=dict(width=3)),
        name='well',
        showlegend=True,
        hovertemplate='Well<br>x=%{x:.0f}m  y=%{y:.0f}m<extra></extra>',
    ), row=1, col=1)

    # --- porosity ---
    hover_poro = np.array(
        [[f'x={x_m[j]:.0f}m  y={y_m[i]:.0f}m<br>φ = {poro[i,j]:.3f}'
          for j in range(nx)] for i in range(ny)]
    )

    fig.add_trace(go.Heatmap(
        z=poro,
        x=x_m, y=y_m,
        colorscale='Plasma',
        zmin=0, zmax=0.45,
        colorbar=dict(title='φ', x=1.01),
        hovertext=hover_poro,
        hoverinfo='text',
        name='poro',
        showlegend=False,
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=x_m[wx], y=y_m[wy],
        mode='markers',
        marker=dict(symbol='x', size=14, color='red', line=dict(width=3)),
        name='well',
        showlegend=False,
        hovertemplate='Well<br>x=%{x:.0f}m  y=%{y:.0f}m<extra></extra>',
    ), row=1, col=2)

    fig.update_layout(
        title=dict(text=f'Field: {field_name}', font=dict(size=15)),
        height=480,
        width=1000,
        legend=dict(x=0.48, y=1.08, orientation='h'),
    )
    fig.update_xaxes(title_text='x (m)')
    fig.update_yaxes(title_text='y (m)', row=1, col=1)

    return fig

# Show all three fields
for name in FIELDS:
    make_figure(name).show()

## 4. Field statistics summary

In [4]:
import pandas as pd

rows = []
for name, (perm, poro, well) in data.items():
    rows.append({
        'field': name,
        'perm_min_mD':    round(perm.min(), 3),
        'perm_geomean_mD': round(float(np.exp(np.log(perm).mean())), 1),
        'perm_max_mD':    round(perm.max(), 1),
        'poro_min':       round(poro.min(), 3),
        'poro_mean':      round(poro.mean(), 3),
        'poro_max':       round(poro.max(), 3),
    })

pd.DataFrame(rows).set_index('field')

,perm_min_mD,perm_geomean_mD,perm_max_mD,poro_min,poro_mean,poro_max
field,,,,,,
homogeneous_100md,100.000,100.0,100.0,0.25,0.250,0.250
fourier_moderate,12.466,100.0,646.6,0.14,0.220,0.328
gmm_3facies,0.089,31.0,16240.5,0.03,0.194,0.407


## 5. Export to standalone HTML

Run this cell to save the three figures as a self-contained HTML file you can share or open in any browser.

In [5]:
from pathlib import Path
import plotly.io as pio

out = Path('outputs/field_explorer.html')
out.parent.mkdir(parents=True, exist_ok=True)

html_parts = []
for i, name in enumerate(FIELDS):
    fig = make_figure(name)
    # include plotly.js only in the first figure
    html_parts.append(fig.to_html(full_html=(i == 0), include_plotlyjs=(i == 0)))

# Stitch together: keep the first file's <html> wrapper, append the others' divs
combined = html_parts[0].replace('</body></html>', '') + '\n'.join(html_parts[1:]) + '</body></html>'
out.write_text(combined, encoding='utf-8')
print(f'Saved: {out}')

Saved: outputs\field_explorer.html
